# Hyperparameter Tuning — K & Threshold

Grid search over retrieval hyperparameters `K` (number of neighbors) and `THRESHOLD` (minimum cosine similarity).

- **Index**: `full` (sbert)
- **Dataset**: IHC
- **Models**: bert, hatebert, roberta
- **Augmentation**: cached once at `MAX_K` / `MIN_THRESHOLD`, then filtered per combo (no redundant encoding)
- **Saved weights**: only the best (K, threshold) per model

**Outputs:**
```
weights_rag_hp_tuning/{model}/k{K}_t{threshold}/
```

## 1. Imports

In [1]:
import os
import json
import shutil
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from rag import encode, retrieve_top_k_above_threshold

## 2. Configuration

Edit `K_VALUES` and `THRESHOLD_VALUES` to define the search grid.

In [2]:
RAG_DIR         = Path('.')
WEIGHTS_RAG_DIR = Path('..') / 'weights_rag_hp_tuning'
INDEX_DIR       = RAG_DIR / 'index'

MODELS = {
    'bert':     'bert-base-uncased',
    'hatebert': 'GroNLP/hateBERT',
    'roberta':  'roberta-base',
}

RETRIEVER_HF_ID = 'sentence-transformers/all-mpnet-base-v2'

SELECTED_MODELS = ['bert', 'hatebert', 'roberta']
INDEX_TYPE      = 'full'
DATASET         = 'IHC'

# Hyperparameter grid
K_VALUES         = [3, 5, 10]
THRESHOLD_VALUES = [0.3, 0.4, 0.5, 0.6]
MAX_K            = max(K_VALUES)
MIN_THRESHOLD    = min(THRESHOLD_VALUES)

# Training config
MAX_LENGTH    = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device            : {device}')
print(f'Retriever         : {RETRIEVER_HF_ID}')
print(f'Models            : {SELECTED_MODELS}')
print(f'Index             : {INDEX_TYPE}')
print(f'Dataset           : {DATASET}')
print(f'K values          : {K_VALUES}')
print(f'Threshold values  : {THRESHOLD_VALUES}')
print(f'Total combos      : {len(K_VALUES) * len(THRESHOLD_VALUES)} per model')

Device            : cuda
Retriever         : sentence-transformers/all-mpnet-base-v2
Models            : ['bert', 'hatebert', 'roberta']
Index             : full
Dataset           : IHC
K values          : [3, 5, 10]
Threshold values  : [0.3, 0.4, 0.5, 0.6]
Total combos      : 12 per model


## 3. Load Datasets

Here we load IHC and/or ISHate and/or Vicomtech depending on the choosen datasets in `SELECTED_DATASETS`.  
Same train/test splits as in `baseline.ipynb`.

In [3]:
# IHC only
raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)

def add_binary_label_ihc(example):
    example['label'] = 0 if example['class'] == 'not_hate' else 1
    return example

train_ihc = splits['train'].map(add_binary_label_ihc)
test_ihc  = splits['test'].map(add_binary_label_ihc)

print(f'IHC — train: {len(train_ihc):,}  test: {len(test_ihc):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Map:   0%|          | 0/19332 [00:00<?, ? examples/s]

Map:   0%|          | 0/2148 [00:00<?, ? examples/s]

IHC — train: 19,332  test: 2,148


## 4. Self-Exclusion Lookup

`chunks_training.csv` maps raw tweet text → `chunk_id` in the FAISS index.
Used at train time to pass `chunk_id` so a model never retrieves itself as a neighbor.

In [4]:
chunks_df = pd.read_csv(RAG_DIR / 'chunks' / 'noihc_chunks_training.csv')

def strip_label_prefix(text):
    return text.replace('[hate] ', '', 1).replace('[not hate] ', '', 1)

text_to_chunk_id = {
    strip_label_prefix(row.text): int(row.chunk_id)
    for _, row in chunks_df.iterrows()
}
print(f'Self-exclusion lookup: {len(text_to_chunk_id):,} entries')

Self-exclusion lookup: 49,049 entries


## 5. Augmentation Functions

Augmentation is **cached once** at `MAX_K` / `MIN_THRESHOLD` to avoid re-encoding for every (K, threshold) combo.
`filter_records` then applies the actual (K, threshold) values in O(N) without any GPU calls.

In [5]:
def augment_split_cached(hf_dataset, text_col, is_train, ret_model, ret_tokenizer, ret_index, ret_documents):
    """Retrieve up to MAX_K neighbors above MIN_THRESHOLD, storing (text, score) pairs for later filtering."""
    records = []
    for example in tqdm(hf_dataset, desc=f"{'train' if is_train else 'test'}"):
        tweet    = example[text_col]
        chunk_id = text_to_chunk_id.get(tweet) if is_train else None
        neighbors = retrieve_top_k_above_threshold(
            tweet, MIN_THRESHOLD, ret_model, ret_tokenizer, ret_index, ret_documents,
            chunk_id=chunk_id, k=MAX_K, use_mean_pool=True,
        )
        records.append({
            'query':     tweet,
            'neighbors': neighbors,   # list of (text, score)
            'label':     example['label'],
        })
    return records


def filter_records(cached, k, threshold):
    """Apply a specific (k, threshold) to cached neighbors — no GPU calls."""
    return [
        {
            'query':     r['query'],
            'neighbors': [text for text, score in r['neighbors'] if score >= threshold][:k],
            'label':     r['label'],
        }
        for r in cached
    ]

## 6. Tokenization

Assembles the augmented string using the model's `sep_token` then tokenizes.

In [6]:
def tokenize_augmented(records, tokenizer, max_length=MAX_LENGTH):
    sep = tokenizer.sep_token
    texts = [
        f' {sep} '.join([r['query']] + r['neighbors'])
        for r in records
    ]
    labels = [r['label'] for r in records]
    encoded = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=max_length,
    )
    encoded['labels'] = labels
    return Dataset.from_dict(encoded)

## 7. Metrics

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

## 8. Hyperparameter Tuning Loop

For each model, sweep all (K, threshold) combos. Only the best combo's weights are saved.

In [ ]:
results = {}

# Load retriever once
print(f"Loading retriever: {RETRIEVER_HF_ID} ...")
ret_tokenizer = AutoTokenizer.from_pretrained(RETRIEVER_HF_ID)
ret_model     = AutoModel.from_pretrained(RETRIEVER_HF_ID).eval().to(device)
print(f"Retriever ready on {device}\n")

# Load full FAISS index once
index_path = INDEX_DIR / 'sbert' / f'vdb_{INDEX_TYPE}.faiss'
ret_index  = faiss.read_index(str(index_path))
with open(INDEX_DIR / f'lookup_{INDEX_TYPE}.json') as f:
    ret_documents = json.load(f)
print(f"Index: {INDEX_TYPE}  |  Vectors: {ret_index.ntotal:,}\n")

# Augment IHC train and test once at MAX_K / MIN_THRESHOLD
print(f"Augmenting IHC train (MAX_K={MAX_K}, MIN_THRESHOLD={MIN_THRESHOLD}) ...")
cached_train = augment_split_cached(train_ihc, 'post', True,
                                    ret_model, ret_tokenizer, ret_index, ret_documents)
print(f"Augmenting IHC test ...")
cached_test  = augment_split_cached(test_ihc,  'post', False,
                                    ret_model, ret_tokenizer, ret_index, ret_documents)

del ret_model, ret_tokenizer
if device.type == 'cuda':
    torch.cuda.empty_cache()
print("Augmentation done. Retriever freed.\n")

# HP tuning loop
for model_name, hf_id in MODELS.items():
    if model_name not in SELECTED_MODELS:
        continue

    print(f"\n{'#'*60}")
    print(f"# Model: {model_name}")
    print(f"{'#'*60}")

    best_f1        = 0.0
    best_k         = None
    best_threshold = None
    best_save_path = str(WEIGHTS_RAG_DIR / model_name / 'best')

    tokenizer = AutoTokenizer.from_pretrained(hf_id)

    for k, threshold in product(K_VALUES, THRESHOLD_VALUES):
        print(f"\n{'='*60}")
        print(f"  K={k}  threshold={threshold}")
        print(f"{'='*60}")

        tok_train = tokenize_augmented(filter_records(cached_train, k, threshold), tokenizer)
        tok_test  = tokenize_augmented(filter_records(cached_test,  k, threshold), tokenizer)

        model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2)

        training_args = TrainingArguments(
            output_dir=f'./checkpoints_hp/{model_name}/k{k}_t{threshold}',
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            learning_rate=LEARNING_RATE,
            eval_strategy='epoch',
            save_strategy='no',
            logging_strategy='epoch',
            report_to='none',
            seed=42,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tok_train,
            eval_dataset=tok_test,
            compute_metrics=compute_metrics,
        )

        trainer.train()

        preds_out = trainer.predict(tok_test)
        preds  = np.argmax(preds_out.predictions, axis=-1)
        labels = [r['label'] for r in filter_records(cached_test, k, threshold)]
        macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)

        print(classification_report(labels, preds, target_names=['Non-HS', 'HS']))

        results[(model_name, k, threshold)] = {
            'macro_f1': macro_f1,
            'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
            'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
        }

        if macro_f1 > best_f1:
            best_f1        = macro_f1
            best_k         = k
            best_threshold = threshold
            # Overwrite best weights dir with current best
            if os.path.exists(best_save_path):
                shutil.rmtree(best_save_path)
            os.makedirs(best_save_path, exist_ok=True)
            trainer.save_model(best_save_path)
            tokenizer.save_pretrained(best_save_path)
            print(f'  *** New best: K={k}, threshold={threshold}, F1={best_f1:.3f} → saved')

        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    # Rename best dir to include the winning combo
    final_save_path = str(WEIGHTS_RAG_DIR / model_name / f'k{best_k}_t{best_threshold}')
    os.rename(best_save_path, final_save_path)
    print(f"\n>>> Best for {model_name}: K={best_k}, threshold={best_threshold}, F1={best_f1:.3f}")
    print(f"    Weights saved → {final_save_path}")

## 9. Results

Full grid: rows = `(model, K, threshold)`, columns = F1 / Precision / Recall. Best F1 per model highlighted.

In [9]:
rows = {}
for (model_name, k, threshold), vals in results.items():
    row_key = f"{model_name} | K={k} | t={threshold}"
    rows[row_key] = {
        'Model': model_name,
        'F1':        vals['macro_f1'],
        'Precision': vals['macro_p'],
        'Recall':    vals['macro_r'],
    }

df = pd.DataFrame(rows).T
df[['F1', 'Precision', 'Recall']] = df[['F1', 'Precision', 'Recall']].astype(float)

# Highlight best F1 per model
def highlight_best(s):
    styles = [''] * len(s)
    for model_name in SELECTED_MODELS:
        mask = df['Model'] == model_name
        if mask.any():
            best_idx = df.loc[mask, 'F1'].idxmax()
            styles[df.index.get_loc(best_idx)] = 'font-weight: bold; background-color: #d4f1d4'
    return styles

styled = (
    df[['F1', 'Precision', 'Recall']]
    .style
    .format('{:.3f}')
    .apply(highlight_best, axis=0)
    .set_caption('HP Tuning — IHC, full index (best per model highlighted)')
)
display(styled)

,F1,Precision,Recall
bert | K=3 | t=0.3,0.784,0.789,0.781
bert | K=3 | t=0.4,0.788,0.793,0.784
bert | K=3 | t=0.5,0.787,0.794,0.782
bert | K=3 | t=0.6,0.784,0.792,0.779
bert | K=5 | t=0.3,0.783,0.788,0.779
bert | K=5 | t=0.4,0.778,0.782,0.776
bert | K=5 | t=0.5,0.784,0.790,0.780
bert | K=5 | t=0.6,0.770,0.783,0.763
bert | K=10 | t=0.3,0.781,0.788,0.776
bert | K=10 | t=0.4,0.781,0.785,0.778
